# Day 3 — Core Expectations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-03-core-expectations.ipynb)

**Course:** Great Expectations for Data Quality  
**Day:** 3 of 5  
**Badge:** Practice

---

## What you will learn

By the end of this notebook you will be able to:

- Run the three fundamental column-level expectations and interpret their results
- Chain multiple expectations on a single Validator and run `validator.validate()`
- Switch between `result_format` modes and understand what each reveals
- Write a reusable `audit_df()` utility that produces a pass/fail table
- Add table-level expectations and test them against a schema violation

## 0 — Setup

In [ ]:
!pip install great-expectations -q

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np

print('great_expectations version:', gx.__version__)

In [ ]:
# Build a synthetic employee DataFrame for all exercises in this notebook
np.random.seed(7)
n = 300

employees = pd.DataFrame({
    'employee_id': range(1, n + 1),
    'name':        [f'Employee_{i}' for i in range(1, n + 1)],
    'department':  np.random.choice(['Engineering', 'Sales', 'HR', 'Finance'], n),
    'age':         np.random.randint(22, 65, n),
    'salary':      np.round(np.random.uniform(40000, 150000, n), 2),
    'email':       [f'emp{i}@corp.example' for i in range(1, n + 1)],
    'start_year':  np.random.randint(2000, 2024, n),
})

# Inject a few nulls and bad values so expectations have something to catch
employees.loc[10:14, 'email'] = None          # 5 null emails
employees.loc[20:22, 'age'] = 999             # 3 impossible ages
employees.loc[30, 'salary'] = -1.0            # 1 negative salary

print('employees shape:', employees.shape)
print(employees.head(4))

In [ ]:
# Set up the GX context and Validator
context = gx.get_context()

datasource = context.data_sources.add_pandas(name='employees_source')
asset = datasource.add_dataframe_asset(name='employees')
batch_request = asset.build_batch_request(dataframe=employees)
suite = context.suites.add(gx.ExpectationSuite(name='employee_suite'))

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=suite,
)

print('Validator ready. Columns:', list(validator.active_batch.data.columns))

## 1 — The Expectations Gallery

GX ships with over 50 built-in expectations. Browse the full catalogue at [greatexpectations.io/expectations](https://greatexpectations.io/expectations/).

Here are 10 expectations you will encounter on real pipelines:

| Expectation | Checks |
|---|---|
| `expect_column_values_to_not_be_null` | No missing values in a column |
| `expect_column_values_to_be_between` | Values fall within `[min_value, max_value]` |
| `expect_column_values_to_match_regex` | Values match a regex pattern |
| `expect_column_values_to_be_in_set` | Values are from an allowed list |
| `expect_column_values_to_be_unique` | No duplicates in a column |
| `expect_column_mean_to_be_between` | Column mean is within a range |
| `expect_column_stdev_to_be_between` | Column standard deviation is within a range |
| `expect_table_row_count_to_be_between` | Row count is within a range |
| `expect_table_columns_to_match_set` | Table has exactly these columns |
| `expect_column_values_to_be_of_type` | Values have a specific data type |

> **Tip:** When you call an `expect_*` method on a Validator, GX does **not** run the check immediately — it appends an `ExpectationConfiguration` to the in-memory suite. Call `validator.validate()` once at the end for efficiency.

> **Reference:** [Expectations reference](https://docs.greatexpectations.io/docs/reference/learn/expectations/)

## 2 — Three Fundamental Column Expectations

We will run the three most common column-level expectations one at a time and inspect the `ExpectationValidationResult` object GX returns.

In [ ]:
# --- Expectation 1: expect_column_values_to_not_be_null ---
# We injected 5 nulls into the 'email' column, so this should FAIL

null_result = validator.expect_column_values_to_not_be_null(
    column='email',
    result_format='SUMMARY',
)

print('expectation_type:', null_result.expectation_config.expectation_type)
print('success:', null_result.success)
print('unexpected_count:', null_result.result.get('unexpected_count'))
print('unexpected_percent:', null_result.result.get('unexpected_percent'))

In [ ]:
# --- Expectation 2: expect_column_values_to_be_between ---
# We injected ages of 999, which are outside the range [18, 80]

age_result = validator.expect_column_values_to_be_between(
    column='age',
    min_value=18,
    max_value=80,
    result_format='SUMMARY',
)

print('expectation_type:', age_result.expectation_config.expectation_type)
print('success:', age_result.success)
print('unexpected_count:', age_result.result.get('unexpected_count'))
print('unexpected_percent:', age_result.result.get('unexpected_percent'))

In [ ]:
# --- Expectation 3: expect_column_values_to_match_regex ---
# Valid emails follow the pattern: word characters, @, domain, dot, tld
# The null emails we injected will count as non-matching

email_result = validator.expect_column_values_to_match_regex(
    column='email',
    regex=r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$',
    result_format='SUMMARY',
)

print('expectation_type:', email_result.expectation_config.expectation_type)
print('success:', email_result.success)
print('unexpected_count:', email_result.result.get('unexpected_count'))
print('partial_unexpected_list:', email_result.result.get('partial_unexpected_list'))

### Reading an ExpectationValidationResult

Every expectation call returns an `ExpectationValidationResult` with this structure:

```
ExpectationValidationResult
  .success              → bool: did the expectation pass?
  .expectation_config   → ExpectationConfiguration: what was checked
  .result               → dict: observed statistics (counts, percents, sample values)
  .exception_info       → dict: filled if GX raised an error during evaluation
  .meta                 → dict: run context metadata
```

The `result` dict keys vary by expectation type, but common ones are:

- `observed_value` — for scalar expectations (`row_count`, `mean`, …)
- `unexpected_count` — number of failing rows
- `unexpected_percent` — percentage of failing rows
- `partial_unexpected_list` — sample of failing values (up to 20 in SUMMARY)

## 3 — Chaining 6 Expectations and Running validate()

In production you define all expectations first, then call `validator.validate()` once. GX runs all checks in a single pass over the data, which is far more efficient than calling each expectation individually and waiting for results.

In [ ]:
# Build a fresh context + validator so the suite starts empty
context2 = gx.get_context()
ds2 = context2.data_sources.add_pandas(name='emp2')
asset2 = ds2.add_dataframe_asset(name='employees2')
br2 = asset2.build_batch_request(dataframe=employees)
suite2 = context2.suites.add(gx.ExpectationSuite(name='chained_suite'))
v2 = context2.get_validator(batch_request=br2, expectation_suite=suite2)

print('Fresh validator ready.')

In [ ]:
# Chain 6 expectations — none run yet; they accumulate in the suite
v2.expect_column_values_to_not_be_null(column='employee_id')
v2.expect_column_values_to_be_unique(column='employee_id')
v2.expect_column_values_to_not_be_null(column='email')
v2.expect_column_values_to_be_between(column='age', min_value=18, max_value=80)
v2.expect_column_values_to_be_between(column='salary', min_value=0, max_value=500000)
v2.expect_column_values_to_be_in_set(
    column='department',
    value_set=['Engineering', 'Sales', 'HR', 'Finance'],
)

print('Expectations staged in suite:', len(v2.get_expectation_suite().expectations))

In [ ]:
# Run ALL expectations in one pass
validation_result = v2.validate()

print('Overall ValidationResult.success:', validation_result.success)
print()
print('Individual results:')
for r in validation_result.results:
    exp_type = r.expectation_config.expectation_type
    col = r.expectation_config.kwargs.get('column', '(table-level)')
    print(f'  [{"PASS" if r.success else "FAIL"}]  {exp_type}  column={col}')

### Overall vs individual success

`ValidationResult.success` is `True` only if **every** expectation in the suite passes. If any single expectation fails, the overall result is `False` — this is the signal your pipeline should use to gate downstream processing.

Individual results (`.results` list) give you the granular view: which checks passed, which failed, and by how much.

## 4 — result_format: BASIC, SUMMARY, COMPLETE

GX lets you control how much detail is returned in each `ExpectationValidationResult` via the `result_format` parameter.

| Mode | What you get |
|---|---|
| `BOOLEAN_ONLY` | `success` flag only — smallest payload |
| `BASIC` | `success` + aggregate stats (counts, percents) |
| `SUMMARY` | BASIC + `partial_unexpected_list` (up to 20 sample values) |
| `COMPLETE` | SUMMARY + full `unexpected_list` and `unexpected_index_list` |

> **Reference:** [result_format reference](https://docs.greatexpectations.io/docs/reference/learn/expectations/result_format)

Let us observe how `unexpected_list` changes across modes.

In [ ]:
# Helper: fresh single-expectation validator
def make_validator(df, suite_name):
    ctx = gx.get_context()
    ds = ctx.data_sources.add_pandas(name=f'ds_{suite_name}')
    a = ds.add_dataframe_asset(name=f'asset_{suite_name}')
    br = a.build_batch_request(dataframe=df)
    s = ctx.suites.add(gx.ExpectationSuite(name=suite_name))
    return ctx.get_validator(batch_request=br, expectation_suite=s)


for fmt in ['BASIC', 'SUMMARY', 'COMPLETE']:
    v_fmt = make_validator(employees, f'fmt_{fmt.lower()}')
    result = v_fmt.expect_column_values_to_be_between(
        column='age',
        min_value=18,
        max_value=80,
        result_format=fmt,
    )
    unexpected_list = result.result.get('unexpected_list')
    partial_list    = result.result.get('partial_unexpected_list')
    print(f'--- result_format={fmt} ---')
    print(f'  unexpected_count:          {result.result.get("unexpected_count")}')
    print(f'  partial_unexpected_list:   {partial_list}')
    print(f'  unexpected_list (len):     {len(unexpected_list) if unexpected_list is not None else "None"}')
    print()

### When to use each format

- **BOOLEAN_ONLY / BASIC** — production pipelines where you only need a pass/fail gate; minimal memory and network overhead
- **SUMMARY** — dashboards and monitoring; enough context to understand *what* failed without flooding logs
- **COMPLETE** — debugging and interactive exploration; the full list of bad rows lets you trace issues to their source

## 5 — Writing audit_df(): A Reusable Pass/Fail Table

A common pattern is to wrap GX validation in a utility that prints a human-readable summary — useful in notebooks, CI logs, and Slack alerts.

In [ ]:
def audit_df(df: pd.DataFrame, validator) -> pd.DataFrame:
    """
    Run all expectations staged on `validator` against `df` and return
    a pass/fail summary DataFrame.

    Columns:
        column         - column name or '(table)' for table-level expectations
        expectation    - short expectation type name
        success        - bool
        unexpected_count - int or None
    """
    validation_result = validator.validate(result_format='SUMMARY')

    rows = []
    for r in validation_result.results:
        col = r.expectation_config.kwargs.get('column', '(table)')
        exp_type = r.expectation_config.expectation_type
        success = r.success
        unexpected_count = r.result.get('unexpected_count')
        rows.append({
            'column':           col,
            'expectation':      exp_type,
            'success':          success,
            'unexpected_count': unexpected_count,
        })

    summary = pd.DataFrame(rows)

    # Pretty-print to stdout
    print(f'=== audit_df report (overall: {"PASS" if validation_result.success else "FAIL"}) ===')
    for _, row in summary.iterrows():
        status = 'PASS' if row['success'] else 'FAIL'
        uc = f"unexpected={row['unexpected_count']}" if row['unexpected_count'] is not None else ''
        print(f'  [{status}]  {row["column"]:<20}  {row["expectation"]:<50}  {uc}')
    print()

    return summary


print('audit_df() defined.')

In [ ]:
# Build a validator with 6 expectations and call audit_df
v_audit = make_validator(employees, 'audit_suite')

v_audit.expect_column_values_to_not_be_null(column='employee_id')
v_audit.expect_column_values_to_be_unique(column='employee_id')
v_audit.expect_column_values_to_not_be_null(column='email')
v_audit.expect_column_values_to_be_between(column='age', min_value=18, max_value=80)
v_audit.expect_column_values_to_be_between(column='salary', min_value=0, max_value=500000)
v_audit.expect_column_values_to_match_regex(
    column='email',
    regex=r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$',
)

audit_summary = audit_df(employees, v_audit)
print('audit_summary DataFrame:')
print(audit_summary)

## 6 — Table-Level Expectations and Schema Violations

Beyond column-level checks, GX offers table-level expectations that validate the structure of the dataset itself. Two essential ones:

- `expect_table_row_count_to_be_between` — guards against empty loads or data explosions
- `expect_table_columns_to_match_set` — enforces schema contracts; catches upstream column renames or drops

In [ ]:
# Validator with table-level expectations on the clean DataFrame
v_table = make_validator(employees, 'table_suite')

expected_columns = {'employee_id', 'name', 'department', 'age', 'salary', 'email', 'start_year'}

v_table.expect_table_row_count_to_be_between(min_value=100, max_value=1000)
v_table.expect_table_columns_to_match_set(column_set=expected_columns)

result_clean = v_table.validate()
print('Clean DataFrame — overall success:', result_clean.success)
for r in result_clean.results:
    print(f'  [{"PASS" if r.success else "FAIL"}]  {r.expectation_config.expectation_type}')

In [ ]:
# Inject a schema violation: rename a column and drop another
broken_df = employees.rename(columns={'salary': 'compensation'}).drop(columns=['start_year'])

print('Broken DataFrame columns:', list(broken_df.columns))
print('(missing: salary, start_year | extra: compensation)')

In [ ]:
# Validate the broken DataFrame against the same suite
v_broken = make_validator(broken_df, 'broken_suite')

v_broken.expect_table_row_count_to_be_between(min_value=100, max_value=1000)
v_broken.expect_table_columns_to_match_set(column_set=expected_columns)

result_broken = v_broken.validate()
print('Broken DataFrame — overall success:', result_broken.success)
for r in result_broken.results:
    status = 'PASS' if r.success else 'FAIL'
    print(f'  [{status}]  {r.expectation_config.expectation_type}')
    if not r.success and 'details' in r.result:
        details = r.result['details']
        print('    mismatched columns:', details)

### Why table-level expectations matter

Column-level expectations silently pass on a broken schema if the column they reference happens to still exist. Table-level expectations catch the structural problems that column checks miss:

- A renamed column (`salary` → `compensation`) makes all column-level checks on `salary` **error out**, not fail cleanly
- `expect_table_columns_to_match_set` catches this immediately with a clear error message
- Run table-level checks **first** in your validation chain so column checks only run on a structurally valid DataFrame

## Challenge — Full Pipeline Audit

Extend the `audit_df()` function and apply it to a new dataset:

1. Create a synthetic `transactions_df` with columns: `txn_id` (unique int), `amount` (float, 0–10000), `currency` (one of 'USD', 'EUR', 'GBP'), `merchant` (str, not null), `status` (one of 'completed', 'pending', 'failed')
2. Inject at least 3 types of violations (null, out-of-range, invalid category)
3. Build a validator with at least 6 expectations covering: null checks, range checks, set membership, uniqueness, row count, and schema
4. Call `audit_df(transactions_df, validator)` and observe the report
5. Modify `audit_df()` to also print the `partial_unexpected_list` for any failing row-level expectation

In [ ]:
# Your solution here
# Step 1: create transactions_df

# Step 2: inject violations

# Step 3: build validator with 6+ expectations

# Step 4: call audit_df

# Step 5: extend audit_df to show partial_unexpected_list


## Day 3 Recap

| Concept | Key takeaway |
|---|---|
| Expectation Gallery | 50+ built-in expectations; browse at greatexpectations.io/expectations |
| `expect_*` staging | Calling `expect_*` appends to the suite; no evaluation until `validate()` |
| `validator.validate()` | Runs all staged expectations in one pass; returns `ValidationResult` |
| `ValidationResult.success` | `True` only if every expectation passes; use this as your pipeline gate |
| `result_format` | BASIC = counts; SUMMARY = + sample failures; COMPLETE = + full failure list |
| `audit_df()` | Utility pattern: wraps `validate()`, returns a tidy pass/fail DataFrame |
| Table-level checks | `row_count` and `columns_to_match_set` guard structure before column checks run |

---

**Up next — Day 4:** Expectation Suites, Checkpoints, and Data Docs — saving your suites to disk, running validation in CI, and generating HTML reports your whole team can read.